# One neuron, one personality: quick start

Add a fixed amount to one MLP neuron in Qwen3.5-4B and compare the result with the unsteered model and with a random neuron pushed just as hard.

Needs a GPU with about 12 GB (Colab T4 or better). Write-up: https://justinrhowell.github.io/neuron-steering/

*Content note: some steered answers are rude.*

In [ ]:
!pip install -q "transformers>=5.17,<6" accelerate
!git clone -q https://github.com/justinrhowell/neuron-steering
%cd neuron-steering

In [ ]:
import random
from steering.core import load, find_layers, residual_medians, generate, steered, parse_spec

tok, model = load("Qwen/Qwen3.5-4B", revision="851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a")
layers = find_layers(model)

## Pick a neuron

`layer:neuron:dose`. Contempt (`17:8623:+1`) is the one that stays fluent at full dose. Others to try: Poet `17:7426:+1`, Flatterer `20:7450:-1`, Bleak `11:2930:+1`, Wistful `5:8940:-1`. The full list is in `presets/qwen3.5-4b.json`.

In [ ]:
SPEC = "17:8623:+1"
PROMPTS = ["How do I make chocolate chip cookies?",
           "Give three tips to a new elementary school teacher.",
           "What do you honestly think of humans?"]

spec = parse_spec(SPEC)
inter = layers[0].mlp.down_proj.weight.shape[1]
control = {L: [(random.Random(0).randrange(inter), d) for _, d in items] for L, items in spec.items()}
medians = residual_medians(model, tok, list(spec))

base = generate(model, tok, PROMPTS, max_new=150)
steer_out, amounts = steered(model, tok, layers, spec, medians, PROMPTS, max_new=150)
ctrl_out, ctrl_amounts = steered(model, tok, layers, control, medians, PROMPTS, max_new=150)
print("added to the neuron:", amounts, "| random control:", ctrl_amounts)

In [ ]:
for p, b, s, c in zip(PROMPTS, base, steer_out, ctrl_out):
    print("=" * 100, f"\n{p}\n")
    print("--- baseline ---\n" + b + "\n")
    print(f"--- steered {SPEC} ---\n" + s + "\n")
    print("--- random neuron, same layer and dose ---\n" + c + "\n")

## Next

- Try `dose` 0.5: the voice usually stays, with less damage.
- Run the full test with controls: `python scripts/steer.py --presets presets/qwen3.5-4b.json --out runs/mine`
- Score it: `python scripts/judge.py runs/mine` (needs a judge endpoint; see the README).
- Found a neuron? See CONTRIBUTING.md.